In [1]:
import lipd as lpd
import pandas as pd
import xarray as xr
import pyleoclim.utils.lipdutils as lipdutils
import numpy as np
from PyAstronomy import pyasl
from datetime import datetime
from collections import OrderedDict
from dateutil import parser

In [7]:
#Open the Lipd file - Note Lipd utils only take full path-Pyleoclim goes around it
d = lpd.readLipd('/Users/deborahkhider/Documents/GitHub/pylipd/Ocn-Vanuatu.Gorman.2012.lpd')
type(d)

Disclaimer: LiPD files may be updated and modified to adhere to standards

reading: Ocn-Vanuatu.Gorman.2012.lpd
Finished read: 1 record


dict

In [9]:
d.keys()

dict_keys(['@context', 'archiveType', 'createdBy', 'dataContributor', 'dataSetName', 'googleDataURL', 'googleMetadataWorksheet', 'googleSpreadSheetKey', 'originalDataURL', 'studyName', 'tagMD5', 'pub', 'geo', 'paleoData', 'lipdVersion'])

Function to extract the information from a LiPD file and create a pandas dataframe.
Note: This is tailored to the data, will need to become more general. 

In [4]:
#def get_dataframe(d):
csv = lpd.getCsv(d)
chronMeasurementTables, paleoMeasurementTables = lipdutils.isMeasurement(csv)
#ignore chron for now.
table_key = paleoMeasurementTables[0].split('.')[-2]
# Measurement dict
paleoData = d['paleoData']['paleo0']['measurementTable'][table_key]['columns']
# Key over to get the headers.
headers = []
val=[]
for item in paleoData.keys():
    if item != 'year': #this would have to be expanded to age or any combination
        headers.append(item)
        val.append(np.array(paleoData[item]['values']))
    elif item == 'year':
        indices = paleoData[item]['values']
        date_time = []
        for item in indices:
            date_time.append(parser.parse(pyasl.decimalYearGregorianDate(item,"yyyy-mm-dd")))
        dti = pd.to_datetime(date_time)
#create Series
series_dict = OrderedDict()
series_dict[d['dataSetName']] = OrderedDict() #handle multiple Lipd files into one object
for idx,item in enumerate(headers):
    series_dict[d['dataSetName']][item] = pd.Series(val[idx],dti)

In [6]:
series_dict

OrderedDict([('Ocn-Vanuatu.Gorman.2012',
              OrderedDict([('d18O',
                            2007-08-17   -4.930000
                            2007-07-17   -5.084667
                            2007-06-17   -5.155617
                            2007-05-17   -5.175080
                            2007-04-17   -5.192504
                                            ...   
                            1842-12-16   -4.647195
                            1842-11-16   -4.643285
                            1842-10-16   -4.595265
                            1842-09-16   -4.423366
                            1842-08-17   -4.120000
                            Length: 1981, dtype: float64),
                           ('d13C',
                            2007-08-17   -2.070000
                            2007-07-17   -2.341584
                            2007-06-17   -2.309383
                            2007-05-17   -2.091698
                            2007-04-17   -2.005719
            

In [ ]:
# warning: pseudocode
class Lipd():
    def __init__(self):
        ### TO DO: Insert above code
        
        ### initialize
        self.lipd_dict=d
        self.dataframe=super(pd.DataFrame(series_dict))